# Applied Activity: From Word Embeddings to Semantic Similarity
### US Immigration Policy — Seven Scopus Abstracts

**Author:** Elián David Martínez Orozco  
**Course:** Fundamentals of AI (with Neural Networks and Transformers)  
**Institution:** Universidad del Norte — Barranquilla, Colombia  
**Professor:** Dr. rer. nat. Humberto Llinás

---

## Corpus: Seven Scopus abstracts on US immigration policy

All abstracts were retrieved from **Scopus** (Elsevier) using the query:
```
"US immigration policy" AND ("border enforcement" OR "visa reform" OR "asylum")
AND (abstract OR research OR paper)
```

| ID | Authors | Title | Journal | Year | DOI |
|----|---------|-------|---------|------|-----|
| P1 | Gonzalez et al. | Structural barriers to prenatal care for migrant women at the US–Mexico border | *Public Health* | 2025 | 10.1016/j.puhe.2025.105986 |
| P2 | Galli | Wolves in Sheep's Clothing? What Central American Unaccompanied Minors Know About Crossing the US-Mexico Border | *Journal of Borderlands Studies* | 2023 | 10.1080/08865655.2023.2200828 |
| P3 | Kerwin | From IIRIRA to Trump: Connecting the Dots to the Current US Immigration Policy Crisis | *Journal on Migration and Human Security* | 2018 | 10.1177/2331502418786718 |
| P4 | Kerwin & Alulema | The CRISIS Survey: The Catholic Church's Work with Immigrants in a Period of Crisis | *Journal on Migration and Human Security* | 2021 | 10.1177/23315024211035726 |
| P5 | Argüellová | So far from god, so close to the US: Current dynamics of Mexican migration | *Central European Journal of International and Security Studies* | 2015 | — |
| P6 | Pantoja | Against the tide? Core American values and attitudes toward US immigration policy | *Journal of Ethnic and Migration Studies* | 2006 | 10.1080/13691830600555111 |
| P7 | Wasem | More than a Wall: The Rise and Fall of US Asylum and Refugee Policy | *Journal on Migration and Human Security* | 2020 | 10.1177/2331502420948847 |

---
## Section 1 — From Frequency-Based to Embedding-Based Representations

**Bag-of-Words** and **TF-IDF** encode text as sparse vectors of token counts or weights. Two abstracts that discuss the same policy topic in different words — one using *deportation*, another using *removal* — would appear dissimilar under TF-IDF because they share no vocabulary. The representation is purely lexical.

**Word embeddings** encode each word as a dense vector learned from contextual usage across large corpora. Words that appear in similar linguistic environments — *deportation*, *removal*, *expulsion* — are mapped to nearby points in vector space, regardless of surface form. This captures **semantic** proximity rather than lexical overlap.

For a corpus of academic abstracts on a specialized topic like immigration policy, this difference is critical: scholars use varied technical vocabulary, paraphrases, and synonyms to discuss the same phenomena. Embedding-based similarity recovers those relationships; TF-IDF misses them.

---
## Section 2 — Embedding Model Description

We use `glove-wiki-gigaword-100`, a **GloVe** model trained on the English Wikipedia and Gigaword 5 corpus.

| Property | Value |
|----------|-------|
| Model | GloVe (Global Vectors for Word Representation) |
| Training corpus | Wikipedia 2014 + Gigaword 5 (~6B tokens) |
| Vocabulary | 400,000 unique tokens |
| Embedding dimension | 100 |
| Access | `gensim.downloader` — automatic download and local cache |

GloVe learns embeddings by factorizing the global word co-occurrence matrix across the entire corpus, as opposed to Word2Vec which uses local context windows. For a formal academic corpus, GloVe's global statistics are generally well-suited.

In [ ]:
%%capture
!pip install gensim pot numpy matplotlib seaborn pandas scikit-learn

In [ ]:
# ── Global imports ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re, unicodedata, warnings
warnings.filterwarnings("ignore")

import gensim
import gensim.downloader as api
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

np.random.seed(42)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False})

print(f"gensim version : {gensim.__version__}")
print(f"numpy version  : {np.__version__}")

# ── Load GloVe model ──────────────────────────────────────────────────────────
print("\nLoading glove-wiki-gigaword-100 (~130 MB, cached after first run)...")
glove = api.load("glove-wiki-gigaword-100")
print(f"Model loaded ✓")
print(f"  Vocabulary size    : {len(glove.key_to_index):,} tokens")
print(f"  Embedding dimension: {glove.vector_size}")

---
## Section 3 — Corpus Definition

Each abstract is stored as a single string, exactly as exported from Scopus. These are the seven documents that will be compared throughout the activity.

In [ ]:
# ── Seven Scopus abstracts on US immigration policy ───────────────────────────
# Source: Scopus CSV export — abstracts reproduced verbatim.

abstracts = {

    "P1_Gonzalez_2025": (
        "Objectives Recent United States (US) immigration policies have left thousands of asylum seekers "
        "stranded in Mexican border cities with limited access to healthcare. Pregnant women are "
        "particularly vulnerable, often facing substantial barriers to receiving adequate prenatal care "
        "(PNC). This study aimed to describe the state of PNC delivery at the US-Mexico border through "
        "the challenges, experiences, and perspectives of healthcare workers (HCWs). Study design "
        "Qualitative study. Methods Semi-structured interviews were conducted with 10 HCWs (6 nurses "
        "and 4 physicians) who volunteered at non-governmental organization (NGO)-operated clinics in "
        "Matamoros and Reynosa between 2019 and 2023. Interviews were conducted in June-July 2023 and "
        "analyzed using thematic analysis. Results Three inductive themes emerged: structural violence; "
        "resource limitations; and care fragmentation. Participants reported that PNC frequently fell "
        "below international standards. Continuity of care was disrupted by patient transience, limited "
        "infrastructure, and reliance on short-term staffing. Additional barriers included shortages of "
        "specialized providers, fragmented medical records, institutional racism, and the impacts of "
        "cartel-related violence. Conclusions This study underscores the structural and systemic barriers "
        "shaping maternal healthcare delivery in humanitarian border settings."
    ),

    "P2_Galli_2023": (
        "This paper examines what Central American unaccompanied minors know about the unauthorized "
        "journey to the US and protective US immigration laws. I find that, contrary to policymakers' "
        "assumptions, children know little about US immigration laws, but they are well aware of the "
        "dangers of the journey. I argue that the composition of migrant networks and the strength of "
        "ties shape how children acquire information and resources indispensable to plan unauthorized "
        "trips to the US, for which most respondents relied on smugglers. Unaccompanied minors who had "
        "strong ties to parents in the US had more access to information and resources than those with "
        "weak ties to non-parent relatives. Yet even strong ties deteriorated after years of family "
        "separation imposed by US immigration policy, undermining communication in families across "
        "borders, with implications for how trips were organized and what children knew. These findings "
        "extend adult-centric migration theories by centering the experiences of children."
    ),

    "P3_Kerwin_2018": (
        "The Illegal Immigration Reform and Immigrant Responsibility Act of 1996 (IIRIRA) has severely "
        "punished US citizens and noncitizens of all statuses. It has also eroded the rule of law by "
        "eliminating due process from the overwhelming majority of removal cases, curtailing equitable "
        "relief from removal, mandating detention without individualized custody determinations for "
        "broad swaths of those facing deportation, and erecting insurmountable roadblocks to asylum. "
        "It created new immigration-related crimes and established the concept of criminal alienhood, "
        "which has slowly but purposefully conflated criminality and lack of immigration status. It also "
        "conditioned family reunification on income, divided mixed-status families, and consigned other "
        "families to marginal and insecure lives in the United States. Finally, it created the 287g "
        "program that enlists state and local law enforcement agencies in immigration enforcement and "
        "drives a wedge between police and immigrant communities."
    ),

    "P4_Kerwin_Alulema_2021": (
        "Over the last five years, the Center for Migration Studies of New York has conducted four "
        "surveys of Catholic immigrant-serving institutions across the United States. The most recent "
        "survey examined how Catholic institutions serving immigrants fared during the combined crises "
        "of the COVID-19 pandemic and the Trump administration's immigration enforcement policies. It "
        "found that Catholic institutions suffered heavy financial and staffing losses during the "
        "pandemic, and that Catholic immigration institutions and their clients were disproportionately "
        "impacted by the pandemic. COVID-19 and the enforcement climate reinforced each other's negative "
        "effects on immigrant communities. Despite these challenges, Catholic institutions demonstrated "
        "remarkable resiliency by pivoting to remote service models, applying for government assistance, "
        "and sustaining their immigrant-serving missions. The survey identified a number of barriers to "
        "immigrants accessing services, including language barriers, fear of immigration enforcement, "
        "and digital divides."
    ),

    "P5_Arguellova_2015": (
        "This work examines the development of US immigration policy with a focus on border enforcement, "
        "migrant removals and the effects on human security at the US-Mexican border. My research "
        "considers three stages in the journey of the unauthorised migrant: clandestine crossing, "
        "detention in the US and deportation to Mexico. Since the border wall was constructed, dynamics "
        "at the border have changed as Mexican and other Latin American migrants have started risking "
        "their lives by crossing in remote areas like deserts and mountains in order to avoid US Border "
        "Patrol and new surveillance technology. Criminal organisations have taken advantage of the "
        "rising interest in human trafficking and begun profiting from the smuggling, robbery and "
        "extortion of migrants, only worsening human security concerns in the area. The militarisation "
        "of the border and increasing protectionism of US immigration policies have been accompanied by "
        "the detention of growing numbers of undocumented migrants, giving rise to a complex detention "
        "system that profits private prisons and detention facilities."
    ),

    "P6_Pantoja_2006": (
        "This article considers the effect of core American values in the structuring of public opinion "
        "toward US immigration policies in the mid-1990s. Using the 1996 American National Election "
        "Study, ordered logistic and logistic analyses are used to examine the impact that individualism, "
        "humanitarianism and egalitarianism have in shaping attitudes toward three policy areas, namely "
        "border enforcement, reductions in the number of immigrants admitted, and immigrant eligibility "
        "for government services. The research finds that the three core values out-performed the "
        "traditional predictors in the third policy area because of the policy's explicit connection to "
        "the welfare state. Core values play a minimal role in shaping attitudes toward border "
        "enforcement. However, the pro-social values - egalitarianism and humanitarianism - are found "
        "to play a key role in favouring increases in the number of immigrants admitted into the US."
    ),

    "P7_Wasem_2020": (
        "This article uses a multidisciplinary approach - analyzing historical sources, refugee and "
        "asylum admissions data, legislative provisions, and public opinion data - to track the rise "
        "and fall of the US asylum and refugee policy. It shows that there has always been a political "
        "struggle between people who advocate for a generous refugee and asylum system and those who "
        "oppose it. Today, the flexible system of protecting refugees and asylees, established in 1980, "
        "is giving way to policies that weaponize them. It offers a historical analysis of US refugee "
        "and asylum policies, as well as xenophobic and nativist attitudes toward refugees. It places "
        "Trump administration refugee policies in three categories: those that abandon longstanding US "
        "legal principles and policies, most notably non-refoulement and due process; those that block "
        "the entry of refugees and asylees; and those that criminalize foreign nationals who attempt to "
        "seek asylum in the United States."
    )
}

doc_keys  = list(abstracts.keys())
doc_texts = list(abstracts.values())

# Short labels for plots
labels = {
    "P1_Gonzalez_2025":      "P1 — Border Healthcare",
    "P2_Galli_2023":          "P2 — Unaccomp. Minors",
    "P3_Kerwin_2018":         "P3 — IIRIRA & Policy",
    "P4_Kerwin_Alulema_2021": "P4 — Catholic Institutions",
    "P5_Arguellova_2015":     "P5 — Border Dynamics",
    "P6_Pantoja_2006":        "P6 — Public Opinion",
    "P7_Wasem_2020":          "P7 — Asylum & Refugees",
}

print("Corpus loaded: 7 abstracts")
print()
for k, v in labels.items():
    n = len(abstracts[k].split())
    print(f"  {v:<30}  ({n} tokens)")

---
## Section 3b — Preprocessing

Academic abstracts from Scopus contain noise that must be removed before NLP analysis.

| Step | Operation | Reason |
|------|-----------|--------|
| 1 | Remove structural headers | `Objectives`, `Methods`, `Results`, `Conclusions` are section labels embedded in P1, not content |
| 2 | Lowercase | Avoids treating `Immigration` and `immigration` as different tokens |
| 3 | Remove diacritics | `Argüellová` → `arguellova`; prevents OOV due to encoding |
| 4 | Remove numbers and special chars | `1996`, `287g`, `©` carry no semantic embedding |
| 5 | Remove stopwords + tokens < 3 chars | Prevents generic English from dominating mean vectors |
| 6 | Strip whitespace | Cosmetic cleanup |

Two versions of the corpus are maintained:
- `abstracts` — raw text, used for display
- `clean_abstracts` — preprocessed text, used for TF-IDF, WMD, and mean embeddings

In [ ]:
# ── Preprocessing pipeline ────────────────────────────────────────────────────
STOPWORDS = set(ENGLISH_STOP_WORDS) | {
    # Abstract structural headers
    "objectives", "methods", "results", "conclusions", "background",
    "design", "findings", "context", "summary", "purpose",
    # Academic filler verbs
    "find", "found", "show", "shows", "argue", "argues",
    "examine", "examines", "consider", "considers", "identify",
    "include", "included", "use", "uses", "used",
    # Common filler
    "also", "however", "thus", "well", "yet", "often",
    "may", "can", "will", "would", "could", "three",
    "four", "five", "six", "ten", "two", "one",
}

_HEADERS = re.compile(
    r"\b(Objectives?|Study\s+design|Methods?|Results?|Conclusions?|"
    r"Background|Introduction|Discussion|Abstract|Summary|Purpose|Findings)\b",
    re.IGNORECASE
)

def preprocess(text: str) -> str:
    """Intelligent cleaning for Scopus academic abstracts."""
    text = _HEADERS.sub(" ", text)                                                           # step 1
    text = text.lower()                                                                       # step 2
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")     # step 3
    text = re.sub(r"[^a-z\s]", " ", text)                                                    # step 4
    tokens = [t for t in text.split() if t not in STOPWORDS and len(t) >= 3]                 # step 5
    return " ".join(tokens)                                                                   # step 6

clean_abstracts = {k: preprocess(v) for k, v in abstracts.items()}
clean_texts     = list(clean_abstracts.values())

# ── Before / After report ─────────────────────────────────────────────────────
print(f"  {'Paper':<30}  {'Raw':>5}  {'Clean':>5}  {'Reduction':>10}")
print("  " + "-" * 57)
for k in doc_keys:
    n_r = len(abstracts[k].split())
    n_c = len(clean_abstracts[k].split())
    print(f"  {labels[k]:<30}  {n_r:>5}  {n_c:>5}  {(n_r-n_c)/n_r*100:>9.1f}%")

print()
print("P1 sample — first 15 clean tokens:")
print(" ".join(clean_abstracts["P1_Gonzalez_2025"].split()[:15]))

---
## Section 4 — Vocabulary Exploration

### 4.1 Select candidate words from the corpus

We select 15 politically and academically relevant words drawn directly from the seven abstracts. The goal is to verify vocabulary coverage before running any similarity analysis — querying an OOV word would raise a `KeyError`.

In [ ]:
# ── Vocabulary coverage check ─────────────────────────────────────────────────
# 15 words drawn from the abstracts — covering multiple sub-themes:
# health, enforcement, economics, legal status, judiciary

candidates = [
    # Health & wellbeing
    "health", "deportation", "mental", "prenatal",
    # Policy & enforcement
    "immigration", "enforcement", "detention", "undocumented",
    # Border & security
    "smuggling", "border", "asylum",
    # Legal & political
    "refugee", "xenophobic", "court",
    # Potentially OOV
    "iirira"
]

in_vocab     = [w for w in candidates if w in glove.key_to_index]
out_of_vocab = [w for w in candidates if w not in glove.key_to_index]

print(f"Candidates   : {len(candidates)}")
print(f"In vocabulary: {len(in_vocab)}")
print(f"OOV          : {len(out_of_vocab)}")
print()
print(f"✅ In vocabulary : {in_vocab}")
print(f"❌ OOV           : {out_of_vocab if out_of_vocab else 'none'}")
print()
print("Why might a word be OOV?")
print("  GloVe was trained on Wikipedia + Gigaword (general text, ~2014).")
print("  Acronyms like 'iirira' do not appear as standalone lowercase tokens")
print("  in general Wikipedia text, making them absent from the vocabulary.")
print("  For specialized corpora, a domain-specific model would be preferable.")

---
## Section 5 — Nearest Neighbors

We query `most_similar()` for three words central to the immigration policy corpus: `immigration`, `deportation`, and `asylum`. The results reveal which concepts the model associates geometrically with these policy terms.

From a political science perspective, these neighbors are informative: they show which semantic fields each term occupies in the embedding space learned from general Wikipedia text.

In [ ]:
# ── Nearest neighbors for three policy keywords ───────────────────────────────
query_words = ["immigration", "deportation", "asylum"]

print("Top-5 nearest neighbors (cosine similarity)")
print("=" * 60)

all_nn = {}
for qw in query_words:
    if qw not in glove.key_to_index:
        print(f"  '{qw}' → OOV")
        continue
    results = glove.most_similar(qw, topn=5)
    all_nn[qw] = results
    print(f"\nQuery: '{qw}'")
    print(f"  {'Neighbor':<20} {'Cosine sim':>12}")
    print("  " + "-" * 34)
    for word, score in results:
        bar = "█" * int(score * 25)
        print(f"  {word:<20} {score:.4f}  {bar}")

print()
print("Interpretation:")
print("  'immigration' → neighbors reflect the policy/reform discourse")
print("  'deportation' → neighbors cluster around enforcement and removal")
print("  'asylum'      → neighbors reflect the humanitarian/refugee frame")

In [ ]:
# ── Visualization: nearest neighbors bar chart ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors = ["#0369a1", "#c8102e", "#15803d"]

for ax, (qw, results), color in zip(axes, all_nn.items(), colors):
    words  = [r[0] for r in results]
    scores = [r[1] for r in results]
    bars   = ax.barh(words[::-1], scores[::-1], color=color, alpha=0.85,
                     edgecolor="white")
    ax.set_title(f"Nearest neighbors: '{qw}'", fontsize=10, pad=8)
    ax.set_xlabel("Cosine similarity")
    ax.set_xlim(0, 1.0)
    for bar, score in zip(bars, scores[::-1]):
        ax.text(score + 0.01, bar.get_y() + bar.get_height()/2,
                f"{score:.3f}", va="center", fontsize=8)

plt.suptitle("GloVe nearest neighbors — US immigration policy vocabulary",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 6 — Analogy Reasoning

We construct two analogy queries grounded in the political and legal context of the abstracts.

### Helper functions

In [ ]:
# ── Helper functions (from course document) ───────────────────────────────────
def in_vocab(w, m):
    """Check if a token is in the vocabulary."""
    return w in m.key_to_index

def require_tokens(m, tokens):
    """Return tokens missing from the vocabulary."""
    return [t for t in tokens if not in_vocab(t, m)]

def analogy(m, positive, negative, topn=3):
    """Execute an analogy query safely."""
    return m.most_similar(positive=positive, negative=negative, topn=topn)

print("Helper functions defined ✓")

In [ ]:
# ── Analogy 1: deportation - enforcement + protection ≈ ?
# Conceptual framing:
# If 'deportation' is the outcome of 'enforcement',
# what is the corresponding outcome of 'protection'?
# Expected: something in the humanitarian/legal status space

pos1 = ["deportation", "protection"]
neg1 = ["enforcement"]
missing1 = require_tokens(glove, pos1 + neg1)

print("Analogy 1: deportation + protection − enforcement ≈ ?")
print(f"  Conceptual logic: if enforcement leads to deportation,")
print(f"  what does protection lead to?")
print(f"  Missing tokens: {missing1 if missing1 else 'none ✓'}")
print()

if not missing1:
    r1 = analogy(glove, positive=pos1, negative=neg1, topn=5)
    print(f"  {'Rank':<5} {'Word':<20} {'Score':>10}")
    print("  " + "-" * 38)
    for i, (w, s) in enumerate(r1, 1):
        print(f"  {i:<5} {w:<20} {s:.4f}")
    print()
    print("  Interpretation: the model situates 'protection' in a legal/rights")
    print("  space that contrasts with the enforcement/removal frame.")

In [ ]:
# ── Analogy 2: undocumented - illegal + worker ≈ ?
# Conceptual framing:
# 'undocumented' encodes legal status; 'illegal' encodes a criminalization framing.
# Replacing the criminal frame with 'worker' should shift toward labor/economic space.
# This analogy probes how the model encodes framing differences in immigration discourse.

pos2 = ["undocumented", "worker"]
neg2 = ["illegal"]
missing2 = require_tokens(glove, pos2 + neg2)

print("Analogy 2: undocumented + worker − illegal ≈ ?")
print(f"  Conceptual logic: shift from criminalization frame to labor frame (P3, P5).")
print(f"  Missing tokens: {missing2 if missing2 else 'none ✓'}")
print()

if not missing2:
    r2 = analogy(glove, positive=pos2, negative=neg2, topn=5)
    print(f"  {'Rank':<5} {'Word':<20} {'Score':>10}")
    print("  " + "-" * 38)
    for i, (w, s) in enumerate(r2, 1):
        print(f"  {i:<5} {w:<20} {s:.4f}")
    print()
    print("  Interpretation: the result reflects the labor/workforce semantic field,")
    print("  consistent with the economic framing of immigration in P5 and P6.")

---
## Section 7 — Word Similarity Matrix

We compute a pairwise cosine similarity matrix for 6 keywords extracted directly from the seven abstracts. This shows the geometric relationships between core concepts in the embedding space.

In [ ]:
# ── Pairwise word similarity matrix ──────────────────────────────────────────
# Six words drawn from the abstracts, one per major sub-theme
selected_words = ["health", "deportation", "smuggling", "asylum", "court", "reform"]

# Verify all are in vocabulary
missing_sw = require_tokens(glove, selected_words)
print(f"Selected words : {selected_words}")
print(f"Missing        : {missing_sw if missing_sw else 'none ✓'}")
print()

# Compute pairwise cosine similarity
n = len(selected_words)
sim_matrix = np.zeros((n, n))

for i, wi in enumerate(selected_words):
    for j, wj in enumerate(selected_words):
        vi = glove[wi]
        vj = glove[wj]
        sim_matrix[i, j] = float(
            np.dot(vi, vj) / (np.linalg.norm(vi) * np.linalg.norm(vj))
        )

df_sim = pd.DataFrame(sim_matrix, index=selected_words, columns=selected_words)

print("Cosine similarity matrix (word-level):")
print(df_sim.round(4).to_string())

In [ ]:
# ── Heatmap visualization ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5.5))

mask = np.eye(n, dtype=bool)   # mask diagonal for visual clarity
sns.heatmap(
    df_sim, annot=True, fmt=".3f", cmap="RdYlGn",
    vmin=0, vmax=1, linewidths=0.5, linecolor="white",
    annot_kws={"size": 9}, ax=ax, cbar_kws={"shrink": 0.7}
)
ax.set_title("Pairwise cosine similarity — immigration policy keywords",
             fontsize=10, pad=10)
ax.tick_params(axis="x", rotation=30, labelsize=9)
ax.tick_params(axis="y", rotation=0, labelsize=9)
plt.tight_layout()
plt.show()

# Most and least similar pairs (excluding diagonal)
sim_flat = df_sim.copy()
np.fill_diagonal(sim_flat.values, np.nan)

max_pair = sim_flat.stack().idxmax()
min_pair = sim_flat.stack().idxmin()
print(f"Most similar pair : {max_pair}  →  {sim_flat.loc[max_pair]:.4f}")
print(f"Least similar pair: {min_pair}  →  {sim_flat.loc[min_pair]:.4f}")
print()
print("Interpretation:")
print(f"  '{max_pair[0]}' and '{max_pair[1]}' occupy nearby regions in the embedding")
print(f"  space — they co-occur in similar contexts in the Wikipedia/Gigaword corpus.")
print(f"  '{min_pair[0]}' and '{min_pair[1]}' are geometrically most distant — their")
print(f"  contextual usage patterns diverge substantially in general text.")

---
## Section 8 — Text Similarity: Mean Embeddings + Cosine Similarity

### Method: mean word embedding

Each abstract is represented as the **arithmetic mean** of the embedding vectors of its tokens:

$$\vec{v}_{\text{abstract}} = \frac{1}{n} \sum_{i=1}^{n} \vec{w}_i$$

Only tokens present in the GloVe vocabulary are included. OOV tokens are silently skipped.

### Helper functions

In [ ]:
# ── Text embedding helpers ────────────────────────────────────────────────────

def normalize_text(s):
    """Lowercase, remove diacritics, strip non-alphabetic characters."""
    s = s.lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")
    s = re.sub(r"[^a-z\s]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def oov_tokens(m, s):
    """Return tokens in s absent from vocabulary m."""
    return [t for t in normalize_text(s).split() if t not in m.key_to_index]

def sent_vector_mean(m, s):
    """Compute mean word embedding for text s (OOV tokens skipped)."""
    toks = [t for t in normalize_text(s).split() if t in m.key_to_index]
    if not toks:
        return None
    return np.mean([m[t] for t in toks], axis=0)

def cosine(u, v):
    """Cosine similarity between two vectors."""
    if u is None or v is None:
        return None
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    if nu == 0 or nv == 0:
        return None
    return float(np.dot(u, v) / (nu * nv))

# ── OOV check on clean abstracts ──────────────────────────────────────────────
print("OOV token check per abstract (using preprocessed text):")
print("-" * 55)
for key in doc_keys:
    oov = oov_tokens(glove, clean_abstracts[key])
    n_tokens = len(normalize_text(clean_abstracts[key]).split())
    pct_oov  = len(oov) / n_tokens * 100
    label = labels[key]
    print(f"  {label:<30}  OOV: {len(oov)}/{n_tokens} ({pct_oov:.1f}%)")
    if oov[:5]:
        print(f"    Sample OOV tokens: {oov[:5]}")

In [ ]:
# ── Compute mean embedding vectors for all 7 abstracts ───────────────────────
# Using clean_abstracts so stopwords do not dilute the mean vector
abstract_vectors = {key: sent_vector_mean(glove, clean_abstracts[key])
                    for key in doc_keys}

print("Mean embedding vectors computed ✓")
print(f"  Each vector has dimension: {abstract_vectors['P1_Gonzalez_2025'].shape[0]}")
print()

# ── Pairwise cosine similarity matrix (7×7) ───────────────────────────────────
n_docs   = len(doc_keys)
sim_docs = np.zeros((n_docs, n_docs))

for i, ki in enumerate(doc_keys):
    for j, kj in enumerate(doc_keys):
        sim_docs[i, j] = cosine(abstract_vectors[ki], abstract_vectors[kj])

short_labels = [labels[k] for k in doc_keys]
df_docs_sim  = pd.DataFrame(sim_docs, index=short_labels, columns=short_labels)

print("Pairwise cosine similarity matrix (abstract-level):")
print(df_docs_sim.round(4).to_string())

In [ ]:
# ── Heatmap — abstract-level similarity ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6.5))

short_labs = [
    "P1\nBorder Health",
    "P2\nMinors",
    "P3\nIIRIRA",
    "P4\nCatholic",
    "P5\nBorder Dyn.",
    "P6\nPublic Opinion",
    "P7\nAsylum"
]

sns.heatmap(
    df_docs_sim.values,
    annot=True, fmt=".4f", cmap="Blues",
    vmin=0.5, vmax=1.0,
    linewidths=0.5, linecolor="white",
    xticklabels=short_labs, yticklabels=short_labs,
    annot_kws={"size": 9}, ax=ax, cbar_kws={"shrink": 0.7}
)
ax.set_title(
    "Mean-embedding cosine similarity — 7 US immigration policy abstracts",
    fontsize=10, pad=10
)
ax.tick_params(axis="x", labelsize=8)
ax.tick_params(axis="y", labelsize=8, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── Identify most and least similar pairs ─────────────────────────────────────
sim_off = df_docs_sim.copy()
np.fill_diagonal(sim_off.values, np.nan)

max_pair = sim_off.stack().idxmax()
min_pair = sim_off.stack().idxmin()

print("Abstract pair similarity ranking:")
print("=" * 65)

# All off-diagonal pairs
pairs = []
for i, ki in enumerate(short_labels):
    for j, kj in enumerate(short_labels):
        if j > i:
            pairs.append((ki, kj, sim_docs[i, j]))

pairs_sorted = sorted(pairs, key=lambda x: x[2], reverse=True)

print(f"  {'Rank':<5} {'Abstract pair':<55} {'Cosine sim':>12}")
print("  " + "-" * 74)
for rank, (a, b, score) in enumerate(pairs_sorted, 1):
    marker = " ← most similar" if rank == 1 else (" ← least similar" if rank == len(pairs_sorted) else "")
    print(f"  {rank:<5} {a[:25]:<27} vs {b[:25]:<27} {score:.4f}{marker}")

print()
top_a, top_b, top_s = pairs_sorted[0]
bot_a, bot_b, bot_s = pairs_sorted[-1]
print(f"Most similar : {top_a} / {top_b}  →  {top_s:.4f}")
print(f"Least similar: {bot_a} / {bot_b}  →  {bot_s:.4f}")

---
## Section 9 — Word Mover's Distance (WMD)

While mean-embedding cosine similarity compares abstracts as single aggregated vectors, **WMD** computes the minimum total cost to transport all words in one abstract to the words in another, using embedding-space distances as the transport cost.

WMD is particularly valuable for academic abstracts because two texts can discuss the same policy topic using entirely different vocabulary — and WMD can detect that semantic proximity even without shared tokens.

**Interpretation:** smaller WMD = more similar.

In [ ]:
# ── Word Mover's Distance between all abstract pairs ─────────────────────────
# Using clean_abstracts: stopwords removed, so transport is between content words only.
# fill_norms() precomputes L2 norms for efficiency.

glove.fill_norms()

norm_texts = {key: normalize_text(clean_abstracts[key]) for key in doc_keys}

wmd_matrix = np.zeros((n_docs, n_docs))

for i, ki in enumerate(doc_keys):
    for j, kj in enumerate(doc_keys):
        if i == j:
            wmd_matrix[i, j] = 0.0
        elif i < j:
            d = glove.wmdistance(norm_texts[ki], norm_texts[kj])
            wmd_matrix[i, j] = d
            wmd_matrix[j, i] = d   # symmetric

df_wmd = pd.DataFrame(wmd_matrix, index=short_labels, columns=short_labels)

print("Word Mover's Distance matrix (lower = more similar):")
print(df_wmd.round(4).to_string())

In [ ]:
# ── WMD heatmap ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Cosine similarity (higher = more similar)
sns.heatmap(
    df_docs_sim.values, annot=True, fmt=".4f", cmap="Blues",
    vmin=0.5, vmax=1.0, linewidths=0.5, linecolor="white",
    xticklabels=short_labs, yticklabels=short_labs,
    annot_kws={"size": 8}, ax=axes[0], cbar_kws={"shrink": 0.7}
)
axes[0].set_title("Cosine similarity (↑ = more similar)", fontsize=10, pad=8)
axes[0].tick_params(axis="x", labelsize=7, rotation=30)
axes[0].tick_params(axis="y", labelsize=7, rotation=0)

# WMD (lower = more similar) — mask diagonal
wmd_plot = df_wmd.copy()
np.fill_diagonal(wmd_plot.values, np.nan)

sns.heatmap(
    wmd_plot.values, annot=True, fmt=".4f", cmap="YlOrRd_r",
    linewidths=0.5, linecolor="white",
    xticklabels=short_labs, yticklabels=short_labs,
    annot_kws={"size": 8}, ax=axes[1], cbar_kws={"shrink": 0.7}
)
axes[1].set_title("Word Mover's Distance (↓ = more similar)", fontsize=10, pad=8)
axes[1].tick_params(axis="x", labelsize=7, rotation=30)
axes[1].tick_params(axis="y", labelsize=7, rotation=0)

plt.suptitle("Text similarity — mean embedding cosine vs Word Mover's Distance",
             fontsize=11, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── WMD ranking + comparison with cosine ─────────────────────────────────────
wmd_pairs = []
cos_dict  = {}

for i, ki in enumerate(doc_keys):
    for j, kj in enumerate(doc_keys):
        if j > i:
            wmd_pairs.append((short_labels[i], short_labels[j],
                              wmd_matrix[i, j], sim_docs[i, j]))

wmd_sorted = sorted(wmd_pairs, key=lambda x: x[2])  # ascending WMD

print("Abstract pairs ranked by WMD (ascending = most similar first):")
print(f"  {'Rank':<5} {'Pair':<52} {'WMD':>8} {'Cosine':>8}")
print("  " + "-" * 78)
for rank, (a, b, wmd, cos) in enumerate(wmd_sorted, 1):
    marker = " ← most similar" if rank == 1 else (" ← least similar" if rank == len(wmd_sorted) else "")
    pair_str = f"{a[:22]} / {b[:22]}"
    print(f"  {rank:<5} {pair_str:<52} {wmd:>8.4f} {cos:>8.4f}{marker}")

print()
print("Agreement check (do WMD and cosine rank the same pair as most similar?):")
cos_pairs_sorted = sorted(wmd_pairs, key=lambda x: x[3], reverse=True)
wmd_top = wmd_sorted[0][:2]
cos_top = cos_pairs_sorted[0][:2]
agree = set(wmd_top) == set(cos_top)
print(f"  WMD most similar   : {wmd_top}")
print(f"  Cosine most similar: {cos_top}")
print(f"  Agreement          : {'✅ YES' if agree else '⚠️ NO — metrics disagree (expected: different geometric principles)'}")

---
## Section 10 — Comparison with TF-IDF

We compute TF-IDF cosine similarity for the same 7 abstracts to directly compare the two approaches.

In [ ]:
# ── TF-IDF cosine similarity ──────────────────────────────────────────────────
# Using clean_texts (preprocessed) so the vocabulary matches the embedding analysis.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vect   = TfidfVectorizer(norm="l2")
tfidf_matrix = tfidf_vect.fit_transform(clean_texts)
sim_tfidf    = cosine_similarity(tfidf_matrix)

df_tfidf_sim = pd.DataFrame(sim_tfidf, index=short_labels, columns=short_labels)

print("TF-IDF cosine similarity matrix:")
print(df_tfidf_sim.round(4).to_string())
print()

# Rank all pairs under TF-IDF
tfidf_pairs = []
for i in range(n_docs):
    for j in range(i+1, n_docs):
        tfidf_pairs.append((short_labels[i], short_labels[j], sim_tfidf[i,j]))

tfidf_sorted = sorted(tfidf_pairs, key=lambda x: x[2], reverse=True)

print("TF-IDF pairs ranked (descending):")
for rank, (a, b, s) in enumerate(tfidf_sorted, 1):
    marker = " ← most similar" if rank==1 else (" ← least similar" if rank==len(tfidf_sorted) else "")
    print(f"  {rank}. {a[:22]} / {b[:22]}   TF-IDF={s:.4f}{marker}")

In [ ]:
# ── Three-way comparison table: TF-IDF vs Cosine(emb) vs WMD ─────────────────
print("Three-way comparison: TF-IDF | Cosine (mean emb) | WMD")
print("=" * 80)
print(f"  {'Pair':<50} {'TF-IDF':>8} {'Cosine':>8} {'WMD':>8}")
print("  " + "-" * 78)

# Rebuild combined table
combined = []
for i in range(n_docs):
    for j in range(i+1, n_docs):
        combined.append((
            f"{short_labels[i][:22]} / {short_labels[j][:22]}",
            sim_tfidf[i,j],
            sim_docs[i,j],
            wmd_matrix[i,j]
        ))

combined_sorted = sorted(combined, key=lambda x: x[1], reverse=True)
for pair, tfidf, cos, wmd in combined_sorted:
    print(f"  {pair:<50} {tfidf:>8.4f} {cos:>8.4f} {wmd:>8.4f}")

print()
print("Key observation:")
print("  TF-IDF similarity is sensitive to SHARED VOCABULARY only.")
print("  Mean-embedding cosine captures SHARED MEANING even across different words.")
print("  WMD provides the finest-grained word-level semantic alignment.")
print("  Pairs that rank differently across the three metrics reveal cases where")
print("  shared meaning exceeds shared vocabulary — the core advantage of embeddings.")

---
## Section 11 — Conceptual Reflection

### What embeddings capture that TF-IDF does not

TF-IDF represents a document as a weighted frequency vector over a fixed vocabulary. Two abstracts that discuss *deportation* and *removal* respectively — synonyms in the immigration context — will share zero TF-IDF score on those tokens unless they happen to use the exact same word. The method is entirely lexical: it counts, it does not understand.

GloVe embeddings encode *distributional context*. Words that appear alongside similar words across millions of Wikipedia sentences end up as nearby vectors. *Deportation*, *removal*, and *expulsion* cluster together not because someone defined them as synonyms, but because their co-occurrence patterns are statistically similar. This is why mean-embedding cosine similarity produces higher off-diagonal values than TF-IDF in this activity: the seven abstracts share semantic territory even when their vocabularies diverge.

### One key limitation of Word2Vec (and GloVe)

Both GloVe and Word2Vec produce **static representations**: each word has exactly one vector regardless of context. The word *border* in P5 refers to a physical enforcement zone in the Sonoran desert; in P6 it refers to an abstract policy object in a public opinion study. Both usages receive the same vector. This context-blindness is the central limitation that motivates **contextual embeddings** (ELMo, BERT, GPT), where the vector for a word changes depending on its sentence-level context.

For the immigration policy corpus in this activity, this limitation is relatively minor because the topic domain is narrow and the word senses are fairly consistent. In a mixed-domain corpus it would matter considerably more.

### When embeddings are preferable

Embeddings are preferable over TF-IDF when:
- The corpus uses technical or domain-specific paraphrases and synonyms heavily
- The goal is **semantic** search or clustering rather than keyword matching
- The corpus is multilingual (multilingual embedding models handle this natively)
- Document length is short — TF-IDF statistics become unstable with very short texts, while embeddings still capture meaning

TF-IDF remains preferable when interpretability is paramount, when the vocabulary overlap between documents is genuinely informative, or when computational resources are limited.

---
## Section 12 — Reproducibility

All code in this notebook is self-contained. No external files are required. All operations are deterministic given `np.random.seed(42)`. Library versions are reported below.

In [ ]:
# ── Library versions ──────────────────────────────────────────────────────────
import sys
import gensim, numpy as np, pandas as pd, matplotlib, seaborn, sklearn

versions = [
    ("Python",     sys.version.split()[0],  "Base language"),
    ("gensim",     gensim.__version__,      "Embeddings, WMD"),
    ("numpy",      np.__version__,          "Numerical operations"),
    ("pandas",     pd.__version__,          "DataFrames"),
    ("matplotlib", matplotlib.__version__,  "Plotting"),
    ("seaborn",    seaborn.__version__,     "Heatmaps"),
    ("scikit-learn",sklearn.__version__,    "TF-IDF baseline"),
]

df_v = pd.DataFrame(versions, columns=["Library", "Version", "Role"])
df_v.index = range(1, len(df_v)+1)
print("Library versions:")
print(df_v.to_string())